# 03 — Risk Classification System (the answer)

The task: *"Build a model to identify students who may be at risk... Define your own
criteria for what makes a student Low / Medium / High Risk and explain your approach."*

Because the EDA proved the features carry no learnable signal (notebook 02) and ML models
cannot beat a majority-class baseline on this data (notebook 04), the deliverable system
is an **explicit criteria-based classifier** with a per-student recommendation engine.

## Risk criteria and justification

| Risk | Criterion | Rationale |
|---|---|---|
| **High** | Total_Score < 60 | Bottom ~20% of the uniform 50–100 range; failing-adjacent performance needing intervention now |
| **Medium** | 60 ≤ Total_Score < 75 | Middle band; below the cohort median, monitorable without immediate escalation |
| **Low** | Total_Score ≥ 75 | Top half-plus; performing at or above expectations |

The criteria are intentionally simple and stated before any modeling: a risk label a
teacher can audit beats a black box trained on noise. Attendance does not enter the label
(EDA showed it carries no information in this dataset) but low attendance (< 60%) is
still surfaced in the recommendation text, since the task's example output includes it.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("dcs_student_data_cleaned.csv")

def assign_risk(total_score):
    if pd.isna(total_score):
        return np.nan
    if total_score < 60:
        return "High"
    if total_score < 75:
        return "Medium"
    return "Low"

df["Risk"] = df["Total_Score"].apply(assign_risk)

print("Risk class distribution:")
print(df["Risk"].value_counts())
print(f"\nTotal_Score range: {df['Total_Score'].min():.1f} - {df['Total_Score'].max():.1f}")

Risk class distribution:
Risk
Low       5010
Medium    3035
High      1955
Name: count, dtype: int64

Total_Score range: 50.0 - 100.0


**Takeaway:** the label is explicit, reproducible, and matches the criteria table exactly.
Distribution is ~20% High / ~30% Medium / ~50% Low — a consequence of Total_Score being
uniform on [50, 100].

## Recommendation engine

Each student's **weakest factor** drives the advice. Columns are **normalised by their
scale before comparison** — `Participation_Score` is on a 0–10 scale while the other
scores are 0–100, so comparing raw values would flag Participation for nearly every
student regardless of their actual profile.

In [2]:
# Column scales differ (Participation is 0-10, the rest 0-100) — normalise before comparing.
factor_max = {
    "Attendance (%)": 100, "Midterm_Score": 100, "Assignments_Avg": 100,
    "Quizzes_Avg": 100, "Participation_Score": 10, "Projects_Score": 100,
    "math_score": 100, "reading_score": 100, "writing_score": 100, "science_score": 100,
}

def get_weakest_factor(row):
    normalised = {k: row[k] / mx for k, mx in factor_max.items() if pd.notna(row[k])}
    return min(normalised, key=normalised.get) if normalised else "General"

def build_recommendation(risk_label, weakest_factor, attendance):
    pretty = weakest_factor.replace("_", " ").replace(" (%)", "")
    if risk_label == "High":
        text = (f"Immediate intervention: targeted remediation in {pretty} "
                f"with weekly tutoring and mentor check-ins.")
    elif risk_label == "Medium":
        text = f"Monitor and reinforce {pretty} through revision schedules and short quizzes."
    else:
        text = f"Performing well; keep momentum and polish {pretty} further."
    if pd.notna(attendance) and attendance < 60:
        text += f" Priority: raise attendance above 60% (currently {attendance:.0f}%)."
    return text

def predict_risk(student_id):
    """Task-required output format for a single student."""
    row = df.loc[df["Student_ID"].astype(str) == str(student_id)].iloc[0]
    weakest = get_weakest_factor(row)
    rec = build_recommendation(row["Risk"], weakest, row["Attendance (%)"])
    print(f"Student: {row['First_Name']} {row['Last_Name']} ({row['Student_ID']})")
    print(f"Attendance: {row['Attendance (%)']:.0f}%")
    print(f"Marks: {row['Total_Score']:.0f}")
    print(f"Risk: {row['Risk'].upper()}")
    print(f"Recommendation: {rec}")

# Demo on the first three students
for sid in df["Student_ID"].head(3):
    predict_risk(sid)
    print()

Student: Omar Williams (S1000)
Attendance: 52%
Marks: 56
Risk: HIGH
Recommendation: Immediate intervention: targeted remediation in science score with weekly tutoring and mentor check-ins. Priority: raise attendance above 60% (currently 52%).

Student: Maria Brown (S1001)
Attendance: 97%
Marks: 51
Risk: HIGH
Recommendation: Immediate intervention: targeted remediation in Projects Score with weekly tutoring and mentor check-ins.

Student: Ahmed Jones (S1002)
Attendance: 57%
Marks: 70
Risk: MEDIUM
Recommendation: Monitor and reinforce math score through revision schedules and short quizzes. Priority: raise attendance above 60% (currently 57%).



**Takeaway:** output matches the task's required format
(`Student / Attendance / Marks / Risk / Recommendation`), the weakest factor is now
scale-corrected, and low attendance is flagged in advice regardless of risk band.

## Bonus — automated report for all students

In [3]:
weakest_factors = [get_weakest_factor(row) for _, row in df.iterrows()]

report = pd.DataFrame({
    "Student": df["First_Name"] + " " + df["Last_Name"],
    "Attendance (%)": df["Attendance (%)"],
    "Risk": df["Risk"],
    "Recommendation": [
        build_recommendation(risk, weak, att)
        for risk, weak, att in zip(df["Risk"], weakest_factors, df["Attendance (%)"])
    ],
})
report.to_csv("student_risk_recommendations.csv", index=False)
print(f"Report rows: {len(report)}")
print("\nWeakest-factor distribution (should vary across students, not collapse to one):")
print(pd.Series(weakest_factors).value_counts())
report.head(10)

Report rows: 10000

Weakest-factor distribution (should vary across students, not collapse to one):
Participation_Score    4134
math_score             2268
science_score           954
reading_score           766
Midterm_Score           592
writing_score           559
Assignments_Avg         197
Attendance (%)          181
Quizzes_Avg             179
Projects_Score          170
Name: count, dtype: int64


,Student,Attendance (%),Risk,Recommendation
0,Omar Williams,52.29,High,Immediate intervention: targeted remediation i...
1,Maria Brown,97.27,High,Immediate intervention: targeted remediation i...
2,Ahmed Jones,57.19,Medium,Monitor and reinforce math score through revis...
3,Omar Williams,95.15,Medium,Monitor and reinforce math score through revis...
4,John Smith,54.18,Medium,Monitor and reinforce math score through revis...
5,Liam Brown,81.08,Medium,Monitor and reinforce math score through revis...
6,Ahmed Jones,57.60,Low,Performing well; keep momentum and polish Part...
7,Ahmed Smith,51.91,Low,Performing well; keep momentum and polish scie...
8,Omar Smith,85.97,Low,Performing well; keep momentum and polish Part...
9,Sara Smith,64.01,Low,Performing well; keep momentum and polish read...


**Takeaway:** `student_risk_recommendations.csv` generated for all 10,000 students with
`Student | Attendance | Risk | Recommendation` — the task's bonus requirement. Risks come
from the explicit criteria (ground truth), not from a model guessing at noise.